In [ ]:
# PreprocessingPipeline -> Answering
import os
from huggingface_hub import InferenceClient
from openai import OpenAI
from huggingface_hub import login
from dotenv import load_dotenv
load_dotenv(override=True)
login(os.getenv("HF_TOKEN"))
hf_client = InferenceClient()
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [62]:
PROMPT = (
    """
You are a scientific assistant. Your role is to understand the query, take the required context retrieved from knowledge base and provide the answer.
If the context is None reply 'There's no information regarding that in the KB' and nothing else.
Understand the prompt, understand the context and also provide easy to understand examples.
Answer in properly formatted markdown"""
)

In [63]:
print(PROMPT)


You are a scientific assistant. Your role is to understand the query, take the required context retrieved from knowledge base and provide the answer.
If the context is None reply 'There's no information regarding that in the KB' and nothing else.
Understand the prompt, understand the context and also provide easy to understand examples.
Answer in properly formatted markdown


In [64]:
def builder_prompt(question, chunks=[]):
    context = "\n\n".join(chunks) if len(chunks) != 0 else "None" 
    prompt = f"""
{PROMPT}

CONTEXT:
{context}

Question:
{question}
    """
    return prompt

In [65]:
print(builder_prompt("What is space?"))



You are a scientific assistant. Your role is to understand the query, take the required context retrieved from knowledge base and provide the answer.
If the context is None reply 'There's no information regarding that in the KB' and nothing else.
Understand the prompt, understand the context and also provide easy to understand examples.
Answer in properly formatted markdown

CONTEXT:
None

Question:
What is space?
    


In [66]:
def answer(prompt):
    if os.getenv("ENV") == "DEV":
        response = hf_client.chat_completion(
                model=os.getenv("ANSWER_MODEL"),
                messages=[
                    {"role": "user", "content": prompt}
                ],
                max_tokens=500,
            )
        return response.choices[0].message.content
    else:
        response = openai_client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "user", "content": prompt}
            ],
            max_tokens=500,
        )
        return response.choices[0].message.content

In [67]:
from pathlib import Path
import sys
sys.path.append(str(Path().resolve().parent))
from pipeline.document_processor import PreprocessDocument

# retrieval
processor = PreprocessDocument("../docs").save_to_chroma()
question = "what is the projection head in SimCLR?"
results = processor.query_kb(question)
chunks = results["documents"][0]

prompt = builder_prompt(question=question, chunks=chunks)
response = answer(prompt)

Loading 2 files with 9 workers


2it [00:10,  5.15s/it]
Batches: 100%|██████████| 3/3 [00:00<00:00,  7.91it/s]


Stored 106 chunks from SimCLR.pdf
Stored 78 chunks from CLRMR.pdf
Total in collection: 184


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14936.67it/s]


In [68]:
print(prompt)



You are a scientific assistant. Your role is to understand the query, take the required context retrieved from knowledge base and provide the answer.
If the context is None reply 'There's no information regarding that in the KB' and nothing else.
Understand the prompt, understand the context and also provide easy to understand examples.
Answer in properly formatted markdown

CONTEXT:
In this work, we introduce a simple framework for contrastive learning of visual representations, which we call _SimCLR_ . Not only does SimCLR outperform previous work (Figure 1), but it is also simpler, requiring neither specialized architectures (Bachman et al., 2019; Hénaff et al., 2019) nor a memory bank (Wu et al., 2018; Tian et al., 2019; He et al., 2019; Misra & van der Maaten, 2019).  
In order to understand what enables good contrastive representation learning, we systematically study the major components of our framework and show that:  
1 Code available at https://github.com/google-research/s

In [86]:
from IPython.display import display, Markdown
display(Markdown(response))

The data augmentation function used in the SimCLR framework for contrastive learning involves a composition of multiple operations. Specifically, the following three augmentations are applied sequentially:

1. **Random Cropping**: This operation randomly crops a portion of the image and then resizes it back to the original size. This helps the model learn invariant features to changes in the spatial configuration of an image.

2. **Random Color Distortions**: This includes changes to the brightness, contrast, saturation, and hue of the images. Such distortions help the model become invariant to color variations in the input images.

3. **Random Gaussian Blur**: This involves applying a Gaussian blur to the image, adding another layer of transformation that simulates blurring effects, which the model needs to learn to ignore.

These augmentations are critical for defining contrastive prediction tasks that yield effective representations. The combination of random cropping and color distortion, in particular, is highlighted as being crucial for achieving good performance.

In [84]:

prompt = builder_prompt(question="what was the data augmentation function used in contrastive learning?", chunks=chunks)
response = answer(prompt)

In [85]:
print(prompt)



You are a scientific assistant. Your role is to understand the query, take the required context retrieved from knowledge base and provide the answer.
If the context is None reply 'There's no information regarding that in the KB' and nothing else.
Understand the prompt, understand the context and also provide easy to understand examples.
Answer in properly formatted markdown

CONTEXT:
In this work, we introduce a simple framework for contrastive learning of visual representations, which we call _SimCLR_ . Not only does SimCLR outperform previous work (Figure 1), but it is also simpler, requiring neither specialized architectures (Bachman et al., 2019; Hénaff et al., 2019) nor a memory bank (Wu et al., 2018; Tian et al., 2019; He et al., 2019; Misra & van der Maaten, 2019).  
In order to understand what enables good contrastive representation learning, we systematically study the major components of our framework and show that:  
1 Code available at https://github.com/google-research/s